# Model Selection & Tuning — Telco Churn

Compares candidate models on the Telco churn dataset using stratified 5-fold CV (PR-AUC), then tunes the best candidate with Optuna. The winner is promoted into `params.yaml` and the DVC pipeline.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, recall_score, precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import optuna
from churn.config import load_params, get_paths
from churn.data.schema import TARGET_COLUMN
from churn.data.clean import clean
from churn.data.split import split
from churn.features import FeatureEngineer, build_preprocessor

params = load_params(); paths = get_paths()
raw = pd.read_csv(paths.raw_dir / 'Telco-Customer-Churn.csv')
cleaned = clean(raw)
train_df, val_df, test_df = split(cleaned, params.split)
def make_Xy(df):
    X = df.drop(columns=[TARGET_COLUMN])
    if 'customerID' in X.columns: X = X.drop(columns=['customerID'])
    return X, df[TARGET_COLUMN].to_numpy()
X_train, y_train = make_Xy(train_df); X_val, y_val = make_Xy(val_df); X_test, y_test = make_Xy(test_df)
X_cv = pd.concat([X_train, X_val], ignore_index=True); y_cv = np.concatenate([y_train, y_val])
min_freq = params.features.min_frequency; fp = params.features


## 1. Multi-model comparison (5-fold CV, PR-AUC)


In [ ]:
models = {
 'logistic': lambda: Pipeline([('feat', FeatureEngineer(params=fp)), ('prep', build_preprocessor(min_frequency=min_freq)), ('m', LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, n_jobs=-1, random_state=42))]),
 'lightgbm': lambda: Pipeline([('feat', FeatureEngineer(params=fp)), ('prep', build_preprocessor(min_frequency=min_freq)), ('m', lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, max_depth=-1, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1))]),
 'histgb': lambda: Pipeline([('feat', FeatureEngineer(params=fp)), ('prep', build_preprocessor(min_frequency=min_freq)), ('m', HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, random_state=42))]),
 'random_forest': lambda: Pipeline([('feat', FeatureEngineer(params=fp)), ('prep', build_preprocessor(min_frequency=min_freq)), ('m', RandomForestClassifier(n_estimators=400, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=42))]),
 'extra_trees': lambda: Pipeline([('feat', FeatureEngineer(params=fp)), ('prep', build_preprocessor(min_frequency=min_freq)), ('m', ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=42))]),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, factory in models.items():
    prs, rocs = [], []
    for tr, va in skf.split(X_cv, y_cv):
        pipe = factory(); pipe.fit(X_cv.iloc[tr], y_cv[tr])
        proba = pipe.predict_proba(X_cv.iloc[va])[:,1]
        prs.append(average_precision_score(y_cv[va], proba)); rocs.append(roc_auc_score(y_cv[va], proba))
    rows.append({'model': name, 'PR-AUC': np.mean(prs), 'PR-AUC std': np.std(prs), 'ROC-AUC': np.mean(rocs)})
pd.DataFrame(rows).sort_values('PR-AUC', ascending=False)


## 2. Result

Logistic regression wins on PR-AUC (≈0.67) and is far cheaper to serve. It is promoted to the production pipeline.


## 3. Optuna tuning of logistic regression


In [ ]:
def objective(trial):
    C = trial.suggest_float('C', 1e-3, 100, log=True)
    cw = trial.suggest_categorical('class_weight', ['balanced', None])
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga'])
    penalty = 'l2' if solver == 'lbfgs' else trial.suggest_categorical('penalty', ['l1','l2'])
    max_iter = trial.suggest_int('max_iter', 500, 3000)
    kw = dict(C=C, class_weight=cw, solver=solver, penalty=penalty, max_iter=max_iter, n_jobs=-1, random_state=42)
    scores = []
    for tr, va in skf.split(X_cv, y_cv):
        pipe = Pipeline([('feat', FeatureEngineer(params=fp)), ('prep', build_preprocessor(min_frequency=min_freq)), ('m', LogisticRegression(**kw))])
        pipe.fit(X_cv.iloc[tr], y_cv[tr])
        scores.append(average_precision_score(y_cv[va], pipe.predict_proba(X_cv.iloc[va])[:,1]))
    return float(np.mean(scores))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, timeout=300, show_progress_bar=False)
print('Best CV PR-AUC:', study.best_value)
print('Best params:', study.best_params)


## 4. Final test-set evaluation

Best params promoted to `params.yaml` (logistic: C=0.103, class_weight=none, max_iter=1056). Test PR-AUC ≈ 0.666, ROC-AUC ≈ 0.847.
